In [ ]:
setwd("splicing")

library(plyr)
library(dplyr)
library(qvalue)
library(msigdbr)
library(data.table)

source("code/enrichment_analysis.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:plyr’:

    arrange, count, desc, failwith, id, mutate, rename, summarise,
    summarize


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




In [3]:
msigdb <- msigdbr(species="Mus musculus", category="C5")
sets <- tapply(msigdb$gene_symbol, INDEX=msigdb$gs_name, FUN=list)
sets <- sets[grep("^GO", names(sets))]

# Ctrl vs. CalTRAP naive

In [4]:
experiment_path <- "data/rmats/batch2/ctrl_vs_naive"

In [5]:
# Combine all splicing events to get background

all_files <- list.files(path=experiment_path, pattern="fromGTF\\.[^.]+\\.txt")

all_events_list <- lapply(all_files, function(x) {
    fread(file.path(experiment_path, x), data.table=FALSE)
})
all_events_df <- do.call(rbind.fill, all_events_list)

In [6]:
# Combine singificant splicing events

signif_files <- list.files(path=experiment_path, pattern="filtered")

signif_events_list <- lapply(signif_files, function(x) {
    df <- fread(file.path(experiment_path, x), data.table=FALSE)
    df$Event_type <- unlist(strsplit(x, split=".", fixed=TRUE))[1]
    df
})
signif_events_df <- do.call(rbind.fill, signif_events_list)

In [7]:
all_genes <- unique(all_events_df$geneSymbol)
print(length(all_genes))

[1] 13845


## All significant events

In [8]:
signif_genes <- unique(signif_events_df$geneSymbol)
print(length(signif_genes))

[1] 1255


In [9]:
enrich_df <- get_enrich(signif_genes, all_genes, sets)
head(enrich_df, 20)

,Pval,P.adj
,<dbl>,<dbl>
GOCC_NEURON_PROJECTION,1.025037e-23,1.065936e-19
GOCC_AXON,1.677894e-20,8.724210e-17
GOCC_SYNAPSE,8.601018e-20,2.981400e-16
GOBP_NEURON_DIFFERENTIATION,8.629347e-15,2.243414e-11
GOBP_CELL_JUNCTION_ORGANIZATION,3.971728e-14,8.260400e-11
GOBP_VESICLE_MEDIATED_TRANSPORT,9.123664e-14,1.581283e-10
GOBP_NEURON_DEVELOPMENT,1.889219e-13,2.806569e-10
GOCC_NEURON_TO_NEURON_SYNAPSE,3.377636e-13,4.390504e-10
GOMF_NUCLEOSIDE_TRIPHOSPHATASE_REGULATOR_ACTIVITY,4.294594e-13,4.962165e-10


## Retained introns

### Negative values corresponds to more retained introns in CalTRAP samples

In [10]:
RI_cal_df <- signif_events_df[(signif_events_df$Event_type == "RI") & (signif_events_df$IncLevelDifference < 0),]
RI_cal_genes <- unique(RI_cal_df$geneSymbol)
print(length(RI_cal_genes))

[1] 69


In [11]:
enrich_RI_cal_df <- get_enrich(RI_cal_genes, all_genes, sets)
head(enrich_RI_cal_df, 20)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_MRNA_PROCESSING,5.663612e-05,0.4973047
GOBP_RNA_PROCESSING,9.564472e-05,0.4973047
GOBP_PROTEIN_MATURATION,5.405277e-04,1.0000000
GOMF_RNA_BINDING,5.823607e-04,1.0000000
GOBP_REGULATION_OF_MRNA_METABOLIC_PROCESS,8.800312e-04,1.0000000
GOBP_MRNA_METABOLIC_PROCESS,9.097294e-04,1.0000000
GOBP_POSITIVE_REGULATION_OF_NUCLEOBASE_CONTAINING_COMPOUND_METABOLIC_PROCESS,9.350787e-04,1.0000000
GOBP_REGULATION_OF_MRNA_PROCESSING,2.121282e-03,1.0000000
GOBP_MITOCHONDRIAL_CYTOCHROME_C_OXIDASE_ASSEMBLY,2.143061e-03,1.0000000


### Positive values corresponds to more retained introns in control samples

In [12]:
RI_ctrl_df <- signif_events_df[(signif_events_df$Event_type == "RI") & (signif_events_df$IncLevelDifference > 0),]
RI_ctrl_genes <- unique(RI_ctrl_df$geneSymbol)
print(length(RI_ctrl_genes))

[1] 25


In [13]:
enrich_RI_ctrl_df <- get_enrich(RI_ctrl_genes, all_genes, sets)
head(enrich_RI_ctrl_df, 20)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_MOTOR_NEURON_MIGRATION,0.005407732,1
GOBP_NEURAL_CREST_CELL_MIGRATION_INVOLVED_IN_AUTONOMIC_NERVOUS_SYSTEM_DEVELOPMENT,0.005407732,1
GOBP_TRNA_THIO_MODIFICATION,0.005407732,1
GOCC_FIBRINOGEN_COMPLEX,0.005407732,1
GOBP_ORGANELLE_LOCALIZATION,0.005413684,1
GOBP_MAINTENANCE_OF_PRESYNAPTIC_ACTIVE_ZONE_STRUCTURE,0.007204063,1
GOMF_STRUCTURAL_CONSTITUENT_OF_PRESYNAPTIC_ACTIVE_ZONE,0.007204063,1
GOBP_RIBOSOME_ASSOCIATED_UBIQUITIN_DEPENDENT_PROTEIN_CATABOLIC_PROCESS,0.008997278,1
GOCC_CYTOSKELETON_OF_PRESYNAPTIC_ACTIVE_ZONE,0.008997278,1


## Skipped exons

### Negative values corresponds to more skipped exons in CalTRAP samples

In [14]:
SE_cal_df <- signif_events_df[(signif_events_df$Event_type == "SE") & (signif_events_df$IncLevelDifference < 0),]
SE_cal_genes <- unique(SE_cal_df$geneSymbol)
print(length(SE_cal_genes))

[1] 559


In [15]:
enrich_SE_cal_df <- get_enrich(SE_cal_genes, all_genes, sets)
head(enrich_SE_cal_df, 20)

,Pval,P.adj
,<dbl>,<dbl>
GOCC_NEURON_PROJECTION,4.452923e-27,4.630595e-23
GOCC_SYNAPSE,3.482557e-25,1.810755e-21
GOCC_AXON,4.027222e-23,1.395969e-19
GOBP_CELL_JUNCTION_ORGANIZATION,1.881614e-17,4.891727e-14
GOCC_POSTSYNAPSE,2.688474e-15,5.591488e-12
GOMF_CYTOSKELETAL_PROTEIN_BINDING,7.733500e-15,1.340344e-11
GOBP_CYTOSKELETON_ORGANIZATION,9.130239e-14,1.356362e-10
GOCC_NEURON_TO_NEURON_SYNAPSE,1.402168e-13,1.822643e-10
GOBP_CELLULAR_COMPONENT_MORPHOGENESIS,9.701129e-13,1.065640e-09


### Positive values corresponds to more skipped exons in control samples

In [16]:
SE_ctrl_df <- signif_events_df[(signif_events_df$Event_type == "SE") & (signif_events_df$IncLevelDifference > 0),]
SE_ctrl_genes <- unique(SE_ctrl_df$geneSymbol)
print(length(SE_ctrl_genes))

[1] 394


In [17]:
enrich_SE_ctrl_df <- get_enrich(SE_ctrl_genes, all_genes, sets)
head(enrich_SE_ctrl_df, 20)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_CELL_JUNCTION_ORGANIZATION,4.443390e-08,0.0004620681
GOCC_SYNAPSE,1.148391e-07,0.0005971059
GOBP_NEURON_DIFFERENTIATION,5.835430e-07,0.0018064853
GOBP_CELL_PART_MORPHOGENESIS,7.446896e-07,0.0018064853
GOBP_NEURON_DEVELOPMENT,1.054712e-06,0.0018064853
GOCC_GOLGI_APPARATUS,1.148747e-06,0.0018064853
GOBP_SYNAPSE_ORGANIZATION,1.216020e-06,0.0018064853
GOCC_NEURON_PROJECTION,2.141592e-06,0.0027151007
GOBP_CELL_MORPHOGENESIS_INVOLVED_IN_NEURON_DIFFERENTIATION,2.676145e-06,0.0027151007


## Mutally exclusive exons

In [18]:
MXE_df <- signif_events_df[signif_events_df$Event_type == "MXE",]
MXE_genes <- unique(MXE_df$geneSymbol)
print(length(MXE_genes))

[1] 146


In [19]:
enrich_MXE_df <- get_enrich(MXE_genes, all_genes, sets)
head(enrich_MXE_df)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_REGULATION_OF_TRANSPORT,3.781070e-06,0.03931935
GOMF_GTPASE_ACTIVATOR_ACTIVITY,1.202734e-05,0.04980134
GOMF_ENZYME_ACTIVATOR_ACTIVITY,1.436715e-05,0.04980134
GOCC_SYNAPSE,6.066969e-05,0.15772603
GOMF_HIGH_VOLTAGE_GATED_CALCIUM_CHANNEL_ACTIVITY,1.305769e-04,0.27157381
GOMF_NUCLEOSIDE_TRIPHOSPHATASE_REGULATOR_ACTIVITY,1.919290e-04,0.33264501


## Alternative 3' splice site

### Negative values corresponds to longer 3' end in CalTRAP samples

In [20]:
A3SS_cal_df <- signif_events_df[(signif_events_df$Event_type == "A3SS") & (signif_events_df$IncLevelDifference < 0),]
A3SS_cal_genes <- unique(A3SS_cal_df$geneSymbol)
print(length(A3SS_cal_genes))

[1] 101


In [21]:
enrich_A3SS_cal_df <- get_enrich(A3SS_cal_genes, all_genes, sets)
head(enrich_A3SS_cal_df)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_REGULATION_OF_SMALL_GTPASE_MEDIATED_SIGNAL_TRANSDUCTION,0.0002519270,1
GOBP_RAS_PROTEIN_SIGNAL_TRANSDUCTION,0.0004205375,1
GOBP_REGULATION_OF_DNA_REPLICATION,0.0009303182,1
GOMF_TRANSCRIPTION_REGULATOR_ACTIVITY,0.0009506683,1
GOBP_SMALL_GTPASE_MEDIATED_SIGNAL_TRANSDUCTION,0.0013271432,1
GOMF_SEQUENCE_SPECIFIC_DNA_BINDING,0.0014333817,1


### Positive values corresponds to longer 3' end in control samples


In [22]:
A3SS_ctrl_df <- signif_events_df[(signif_events_df$Event_type == "A3SS") & (signif_events_df$IncLevelDifference > 0),]
A3SS_ctrl_genes <- unique(A3SS_ctrl_df$geneSymbol)
print(length(A3SS_ctrl_genes))

[1] 88


In [23]:
enrich_A3SS_ctrl_df <- get_enrich(A3SS_ctrl_genes, all_genes, sets)
head(enrich_A3SS_ctrl_df)

,Pval,P.adj
,<dbl>,<dbl>
GOMF_TFIIB_CLASS_TRANSCRIPTION_FACTOR_BINDING,0.0002376811,1
GOBP_NEURON_DIFFERENTIATION,0.0003403521,1
GOBP_POSITIVE_REGULATION_OF_RNA_METABOLIC_PROCESS,0.0004847589,1
GOBP_POSTTRANSCRIPTIONAL_REGULATION_OF_GENE_EXPRESSION,0.0006416371,1
GOBP_POSITIVE_REGULATION_OF_NUCLEOBASE_CONTAINING_COMPOUND_METABOLIC_PROCESS,0.0006567793,1
GOBP_NEURON_DEVELOPMENT,0.0007376079,1


## Alternative 5' splice site

### Negative values corresponds to longer 5' end in CalTRAP samples

In [24]:
A5SS_cal_df <- signif_events_df[(signif_events_df$Event_type == "A5SS") & (signif_events_df$IncLevelDifference < 0),]
A5SS_cal_genes <- unique(A5SS_cal_df$geneSymbol)
print(length(A5SS_cal_genes))

[1] 49


In [25]:
enrich_A5SS_cal_df <- get_enrich(A5SS_cal_genes, all_genes, sets)
head(enrich_A5SS_cal_df)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_POSITIVE_REGULATION_OF_DENDRITIC_SPINE_DEVELOPMENT,0.0001922430,1
GOBP_REGULATION_OF_CIRCADIAN_SLEEP_WAKE_CYCLE,0.0002547908,1
GOBP_CIRCADIAN_SLEEP_WAKE_CYCLE,0.0005422862,1
GOMF_NEUREXIN_FAMILY_PROTEIN_BINDING,0.0005422862,1
GOBP_REGULATION_OF_DENDRITIC_SPINE_DEVELOPMENT,0.0006442351,1
GOMF_TRANSCRIPTION_FACTOR_BINDING,0.0007181913,1


### Positive values corresponds to longer 5' end in control samples


In [26]:
A5SS_ctrl_df <- signif_events_df[(signif_events_df$Event_type == "A5SS") & (signif_events_df$IncLevelDifference > 0),]
A5SS_ctrl_genes <- unique(A5SS_ctrl_df$geneSymbol)
print(length(A5SS_ctrl_genes))

[1] 52


In [27]:
enrich_A5SS_ctrl_df <- get_enrich(A5SS_ctrl_genes, all_genes, sets)
head(enrich_A5SS_ctrl_df)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_OXIDATIVE_RNA_DEMETHYLATION,8.261833e-05,0.4295740
GOMF_OXIDATIVE_RNA_DEMETHYLASE_ACTIVITY,8.261833e-05,0.4295740
GOBP_PEPTIDYL_LYSINE_MODIFICATION,1.427824e-04,0.4949314
GOBP_NEURON_DEVELOPMENT,3.550625e-04,0.9215353
GOBP_PEPTIDYL_LYSINE_HYDROXYLATION,4.897763e-04,0.9215353
GOBP_CELL_RECOGNITION,5.317061e-04,0.9215353
